In [0]:
catalog_name = "crypto"
schema_name = "bronze"
schema_silver = "silver"
volume_name = "binance_raw_data"

"""
Creation <catalogue>.<schema>.<volume> if not exists
"""

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog_name}.{schema_name}.{volume_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_silver}")

display(spark.sql(f"DESCRIBE VOLUME {catalog_name}.{schema_name}.{volume_name}"))

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")
spark.sql(f"USE SCHEMA {schema_name}")

display(spark.sql(f"SHOW TABLES"))

In [0]:
spark.catalog.setCurrentCatalog(catalog_name)
spark.catalog.setCurrentDatabase(schema_name)

# Display available tables in your schema
spark.catalog.listTables(schema_name)

In [0]:
%sql
SHOW VOLUMES IN crypto.bronze;

In [0]:
%sql
USE CATALOG crypto;
USE SCHEMA bronze;

In [0]:
spark.sql(f"LIST '/Volumes/{catalog_name}/{schema_name}/{volume_name}/' ").display()

In [0]:
path_daily = "/Volumes/crypto/bronze/binance_raw_data/batch/BTCUSDT/*/*.csv"


spark.sql(f"DROP TABLE IF EXISTS crypto.bronze.crypto")

spark.sql(f"""
CREATE OR REPLACE TABLE crypto.bronze.crypto
AS
SELECT 
    -- Conversion du timestamp Binance (ms) en Timestamp Spark
    split(_metadata.file_path, '/')[6] AS symbol,
    FROM_UNIXTIME(open_time / 1000) AS open_time,
    split(_metadata.file_path, '/')[7] AS timeframe,
    CAST(open AS DOUBLE) AS open_price,
    CAST(high AS DOUBLE) AS high_price,
    CAST(low AS DOUBLE) AS low_price,
    CAST(close AS DOUBLE) AS close_price,
    CAST(volume AS DOUBLE) AS volume,
    FROM_UNIXTIME(close_time / 1000) AS close_time,
    CAST(quote_volume AS DOUBLE) AS quote_volume,
    CAST(count AS INT) AS trade_count,
    CAST(taker_buy_volume AS DOUBLE) AS taker_buy_volume,
    CAST(taker_buy_quote_volume AS DOUBLE) AS taker_buy_quote_volume
FROM read_files(
    "/Volumes/crypto/bronze/binance_raw_data/batch/*/*/*.csv",
    format => 'csv',
    header => true,
    inferSchema => true

);
""")

display(spark.table("crypto.bronze.crypto").head(10))




In [0]:
'''
spark.sql(f"DROP TABLE IF EXISTS crypto.bronze.crypto_1h")

spark.sql(f"""
CREATE OR REPLACE TABLE crypto.bronze.crypto_1h
AS
SELECT 
    -- Convert Timestamp Binance (ms) to Timestamp Spark
    FROM_UNIXTIME(open_time / 1000) AS open_time,
    CAST(open AS DOUBLE) AS open_price,
    CAST(high AS DOUBLE) AS high_price,
    CAST(low AS DOUBLE) AS low_price,
    CAST(close AS DOUBLE) AS close_price,
    CAST(volume AS DOUBLE) AS volume,
    FROM_UNIXTIME(close_time / 1000) AS close_time,
    CAST(quote_volume AS DOUBLE) AS quote_volume,
    CAST(count AS INT) AS trade_count,
    CAST(taker_buy_volume AS DOUBLE) AS taker_buy_volume,
    CAST(taker_buy_quote_volume AS DOUBLE) AS taker_buy_quote_volume,
    split(_metadata.file_path, '/')[6] AS symbol
FROM read_files(
    "/Volumes/crypto/bronze/binance_raw_data/batch/*/1h/*.csv",
    format => 'csv',
    header => true,
    inferSchema => true
);
""")

display(spark.table("crypto.bronze.crypto_1h").head(10))
'''

In [0]:
spark.sql(f"SHOW TABLES;").display()

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW tempview_symbol AS
SELECT
  symbol,
  count(*) as count_symbol,
  timeframe,
  count(*) AS count_tf
FROM crypto.bronze.crypto
GROUP BY symbol, timeframe;
    
SELECT * FROM tempview_symbol

In [0]:
%sql
USE CATALOG crypto;
USE SCHEMA silver;

In [0]:
#### ERROR HERE : Impossible to create cadle_pk as primary key
#### Because symbole is STRING but can be null 
#### No fix seems pushed databricks

"""
spark.sql(f"DROP TABLE IF EXISTS crypto.silver.crypto_1d")

spark.sql(f"""
CREATE OR REPLACE TABLE crypto.silver.candle_1d
AS
SELECT
    CAST(symbol AS STRING),
    CAST(open_time AS TIMESTAMP),
    CAST(open_price AS DOUBLE),
    CAST(high_price AS DOUBLE),
    CAST(low_price AS DOUBLE),
    CAST(close_price AS DOUBLE),
    CAST(close_time AS TIMESTAMP)
FROM crypto.bronze.crypto_1d
WHERE
    symbol IS NOT NULL
    AND open_time IS NOT NULL
    
    AND high_price IS NOT NULL
    AND low_price IS NOT NULL
    AND close_price IS NOT NULL
    AND close_time IS NOT NULL
""") 

spark.sql("""
ALTER TABLE crypto.silver.candle_1d
ADD CONSTRAINT candles_pk PRIMARY KEY(symbol, open_time)
""")
"""

In [0]:
### Candlestick data ###

spark.sql(f"DROP TABLE IF EXISTS crypto.silver.candle")

spark.sql("""
CREATE OR REPLACE TABLE crypto.silver.candle (
    symbol STRING NOT NULL,
    open_time TIMESTAMP NOT NULL,
    timeframe STRING NOT NULL,
    open_price DOUBLE,
    high_price DOUBLE,
    low_price DOUBLE,
    close_price DOUBLE,
    close_time TIMESTAMP,
    CONSTRAINT candle_pk PRIMARY KEY(symbol, open_time)
)
USING DELTA
PARTITIONED BY (symbol)
""")

In [0]:
### Metrics table ###

spark.sql(f"DROP TABLE IF EXISTS crypto.silver.metrics_ohlcv")

spark.sql("""
CREATE OR REPLACE TABLE crypto.silver.metrics_ohlcv (
    symbol STRING NOT NULL,
    open_time TIMESTAMP NOT NULL,
    timeframe STRING NOT NULL,
    volume DOUBLE,
    quote_volume DOUBLE,
    trade_count INT,
    taker_buy_volume DOUBLE,
    taker_buy_quote_volume DOUBLE,
    CONSTRAINT metric_pk PRIMARY KEY(symbol, open_time),
    FOREIGN KEY (symbol, open_time) REFERENCES crypto.silver.candle(symbol, open_time)
)
USING DELTA 
PARTITIONED BY (symbol)
""")


In [0]:
spark.sql(f"""
          SELECT COUNT(*) AS null_count
          FROM crypto.bronze.crypto
          WHERE symbol IS NULL
          AND open_time IS NULL
          """).show()

In [0]:
spark.sql("""
          INSERT INTO crypto.silver.candle
          SELECT 
              symbol,
              open_time,
              timeframe,
              open_price,
              high_price,
              low_price,
              close_price,
              close_time
          FROM crypto.bronze.crypto
          WHERE
              symbol IS NOT NULL
              AND open_time IS NOT NULL;
          
          """)

In [0]:
display(spark.table("crypto.silver.candle").head(10))


In [0]:
spark.sql("""
          INSERT INTO crypto.silver.metrics_ohlcv
          SELECT 
              symbol,
              open_time,
              timeframe,
              volume,
              quote_volume,
              trade_count,
              taker_buy_volume,
              taker_buy_quote_volume
          FROM crypto.bronze.crypto
          WHERE symbol IS NOT NULL 
          AND open_time IS NOT NULL;
          """)

display(spark.table("crypto.silver.metrics_ohlcv").head(10))